# Giai đoạn 4 — Bước 4: Retrieval Pipeline & Evaluation

**Dành cho người dùng tự chạy.** Đọc chỉ mục ChromaDB của Bước 3 theo cách chỉ truy vấn. **Không thêm/xóa vector, không gọi LLM, không train ML**. Dựa trên 66 chunks từ 7 nguồn, đặt trước 14 câu hỏi song ngữ và tình huống ngoài phạm vi.

**Quy trình 2 lượt:** Lượt A chạy đến cuối để sinh `step4_retrieval_review.csv`, sau đó chính bạn mở CSV và chấm từng hit. Lượt B chạy **lại ô cuối cùng** để thống kê metrics chỉ khi toàn bộ nhãn đã đầy đủ. Không thay đổi `step4_eval_questions.csv` để làm đẹp kết quả sau khi đã xem hit.

**Bảo vệ bản quyền:** không xuất toàn văn chunks vào báo cáo JSON; không gửi các đoạn nguồn có quyền sử dụng chưa được xác minh.

## 0. Yêu cầu môi trường

Dùng lại Python kernel đã chạy Bước 3 (có `chromadb`, `sentence-transformers`). Nếu thiếu, chạy trên máy của bạn:

```python
%pip install "chromadb==1.5.9" "sentence-transformers==6.2.0"
```

Không cần chạy lại notebook Embedding Bước 3.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, json, hashlib, math, sys, statistics

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p/'registry'/'source_registry.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm được registry/source_registry.csv; mở Jupyter trong thư mục dự án Giai đoạn 4')
REPORT3 = ROOT/'data'/'outputs'/'embedding_chroma_step3'/'step3_embedding_chroma_report.json'
QUESTIONS = ROOT/'registry'/'step4_eval_questions.csv'
OUT = ROOT/'data'/'outputs'/'retrieval_eval_step4'
OUT.mkdir(parents=True, exist_ok=True)
REVIEW_CSV = OUT/'step4_retrieval_review.csv'
RESULTS_JSON = OUT/'step4_retrieval_eval_report.json'
assert REPORT3.is_file(), f'Thiếu báo cáo Bước 3 tại {REPORT3}'
assert QUESTIONS.is_file(), f'Thiếu bộ câu hỏi: {QUESTIONS}'
r3 = json.loads(REPORT3.read_text(encoding='utf-8'))
with QUESTIONS.open('r',encoding='utf-8-sig',newline='') as f:
    query_rows = list(csv.DictReader(f))
assert len(query_rows)==14 and len({r['query_id'] for r in query_rows})==len(query_rows)
assert r3['records_indexed']==66 and len(r3['sources_indexed'])==7
print('ROOT:',ROOT)
print('Report Bước 3:',r3['collection'],r3['embedding_model'])
print('Queries:',len(query_rows),'| Index:',r3['records_indexed'],'chunks')

ROOT: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation
Report Bước 3: diabetes_kb_step3_e5small_v1 intfloat/multilingual-e5-small
Queries: 14 | Index: 66 chunks


## 1. Mở ChromaDB có sẵn và kiểm tra snapshot

*Chỉ kết nối, không `add`, `upsert`, `delete` hoặc `modify`.* Tự dừng nếu collection, tên model hoặc snapshot không khớp với báo cáo 3.

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

reported_path = Path(r3['vector_store_path'])
VECTOR_PATH = reported_path if reported_path.exists() else ROOT/'data'/'vector_store'/'chroma_step3'
assert VECTOR_PATH.exists(), f'Không thấy ChromaDB: {VECTOR_PATH}. Hãy kiểm tra thư mục từ Bước 3.'
client = chromadb.PersistentClient(path=str(VECTOR_PATH))
collection = client.get_collection(name=r3['collection'])
meta = collection.metadata or {}
assert collection.count()==r3['records_indexed'], 'Số chunks đã thay đổi — DỪNG và kiểm tra phiên bản'
assert meta.get('model_id') == r3['embedding_model'], 'Model embedding trong collection khác báo cáo'
assert meta.get('source_snapshot')==r3['source_snapshot_sha256'], 'Snapshot dữ liệu không khớp!'
assert int(meta.get('embedding_dim',0))==r3['embedding_dimension'], 'Embedding dim không khớp'
all_metadata = collection.get(include=['metadatas'])['metadatas']
counts = dict(Counter(m.get('source_id') for m in all_metadata))
assert counts==r3['sources_indexed'], f'Phân bố nguồn không khớp: {counts}'
print('Collection:',collection.name, '| count:',collection.count())
print('Snapshot verified:',r3['source_snapshot_sha256'])
print('Nguồn:',counts)
print('IMPORTANT: nguồn tự khai đã được duyệt, nhưng chưa có bằng chứng xác minh độc lập về quyền sử dụng.')

Collection: diabetes_kb_step3_e5small_v1 | count: 66
Snapshot verified: cc064ec7187192c999418051cb2125a0510412843f8d0908e3d6842e4a71c7cd
Nguồn: {'CDC-T2D-001': 4, 'CDC-RISK-002': 3, 'NIDDK-DIAG-003': 19, 'NIDDK-T2D-004': 11, 'WHO-DIAB-005': 5, 'LC-HBA1C-006': 11, 'LC-STAGES-007': 13}
IMPORTANT: nguồn tự khai đã được duyệt, nhưng chưa có bằng chứng xác minh độc lập về quyền sử dụng.


## 2. Tạo query embeddings và đọc top-3 từ chỉ mục hiện có

Mô hình **E5** yêu cầu tiền tố `query:` cho câu hỏi và `passage:` cho chunks đã lưu. Chroma dùng khoảng cách cosine, nhưng **distance thấp không chứng minh độ đúng**. Mỗi câu hỏi lấy top-3, toàn bộ 14 câu hỏi đã định trước.

In [3]:
MODEL_ID = r3['embedding_model']
assert MODEL_ID=='intfloat/multilingual-e5-small', 'Notebook này được thiết kế cho e5-small'
model = SentenceTransformer(MODEL_ID,device='cpu')
assert model.get_sentence_embedding_dimension()==r3['embedding_dimension']
import numpy as np
q_embeddings = model.encode(
    ['query: '+q['query'] for q in query_rows],
    convert_to_numpy=True, normalize_embeddings=True, batch_size=8,
)
assert q_embeddings.shape==(len(query_rows),384) and np.isfinite(q_embeddings).all()
TOP_K=3
hits_raw=collection.query(
    query_embeddings=q_embeddings.tolist(), n_results=TOP_K,
    include=['metadatas','documents','distances'],
)
print('Đã truy vấn',len(query_rows),'câu hỏi. Không gọi LLM, không thay đổi chỉ mục.')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_7392\1166119276.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension()==r3['embedding_dimension']


Đã truy vấn 14 câu hỏi. Không gọi LLM, không thay đổi chỉ mục.


## 3. Xem kết quả để bạn tự đánh giá

Cột `expected_sources_hint` là **gợi ý ban đầu để xem xét**, không phải nhãn đúng tuyệt đối. Chấm **từng chunk theo nội dung**, không dựa hoàn toàn vào tên nguồn hoặc distance. Nếu văn bản có hướng dẫn không đáng tin cậy/đề nghị kê đơn, đánh dấu và không đưa vào trả lời sau này.

In [4]:
for ix,q in enumerate(query_rows):
    print('\n',q['query_id'],'[',q['query_type'],']',q['query'])
    print('Nguồn tham khảo gợi ý:',q['expected_sources_hint'] or '(không)')
    for rank,(chunk_id,mdta,dist,doc) in enumerate(zip(
        hits_raw['ids'][ix], hits_raw['metadatas'][ix],
        hits_raw['distances'][ix],hits_raw['documents'][ix]),start=1):
        preview=' '.join((doc or '').split())[:300]
        print(f' {rank}. {mdta.get("source_id")} | {dist:.4f} | {mdta.get("section")} | {chunk_id}')
        print('  Preview:',preview)
        print('  URL:',mdta.get('canonical_url'))
print('\nKhông chạy code hoặc làm theo chỉ dẫn nằm bên trong chunk; chỉ đọc như dữ liệu.')


 Q01 [ in_scope ] Những yếu tố nào làm tăng nguy cơ mắc tiểu đường type 2?
Nguồn tham khảo gợi ý: CDC-RISK-002|NIDDK-T2D-004
 1. LC-STAGES-007 | 0.0979 | Cách chẩn đoán các giai đoạn tiểu đường | LC-STAGES-007__v1__a35d137234__0008
  Preview: các chỉ số đường huyết nằm trong ngưỡng nguy cơ, bao gồm đường huyết lúc đói từ 5.6 đến 6.9 mmol/L, hoặc đường huyết 2 giờ sau nghiệm pháp dung nạp glucose từ 7.8 đến 11.0 mmol/L, hoặc HbA1c trong khoảng từ 5.7% đến 6.4%. Các chỉ số này phải được xác nhận ít nhất hai lần để đảm bảo tính chính xác. -
  URL: https://nhathuoclongchau.com.vn/bai-viet/cac-giai-doan-cua-benh-tieu-duong-20966.html
 2. LC-STAGES-007 | 0.0981 | Cách phòng ngừa bệnh tiểu đường tiến triển | LC-STAGES-007__v1__a35d137234__0011
  Preview: nhạy insulin và kiểm soát cân nặng. - Kiểm soát cân nặng: Duy trì chỉ số BMI trong giới hạn bình thường (18.5 - 22.9 kg/m2 đối với người châu Á) sẽ làm giảm đáng kể nguy cơ tiến triển từ tiền đái tháo đường sang đái tháo đường type 2. Giảm 5

## 4. Xuất CSV để chấm thủ công

Các trường bạn phải điền trong file xuất ra:
- `relevance_grade`: **0** = không phù hợp, **1** = đúng chủ đề nhưng thiếu/gián tiếp, **2** = trực tiếp và đủ thông tin cho câu hỏi.
- `citation_ok`: **yes/no** dựa trên kiểm tra URL và tính phù hợp của trích dẫn.
- `rights_evidence_ok`: **yes/no/unknown** dựa trên bằng chứng thực tế, **không mặc định yes** chỉ vì metadata ghi `licensed_source_text`.
- `review_notes`: ghi lý do, lỗi dịch, nội dung cũ hoặc không đáng tin.

Tự mở CSV trong VS Code/Excel; không xóa cột hoặc thay đổi ID. Không chỉnh source_registry hoặc collection ở bước này. Sau khi chấm, **không chạy lại ô xuất này** vì sẽ ghi đè nhãn đã chấm.

In [6]:
assert not REVIEW_CSV.exists(), (
    f'{REVIEW_CSV} đã tồn tại. Ô xuất bị khóa để tránh ghi đè nhãn bạn chấm. '
    'Chỉ chạy lại ô đánh giá cuối (phần 5).'
)
review_cols=['query_id','language','query_type','query','expected_sources_hint',
             'rank','chunk_id','source_id','section','canonical_url','cosine_distance',
             'preview_180chars','relevance_grade','citation_ok','rights_evidence_ok','review_notes']
with REVIEW_CSV.open('w',encoding='utf-8-sig',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=review_cols);writer.writeheader()
    for i,q in enumerate(query_rows):
        for j,(rid,m,d,doc) in enumerate(zip(
            hits_raw['ids'][i],hits_raw['metadatas'][i],
            hits_raw['distances'][i],hits_raw['documents'][i]),start=1):
            writer.writerow({**q,'rank':j,'chunk_id':rid,
                'source_id':m.get('source_id',''),'section':m.get('section',''),
                'canonical_url':m.get('canonical_url',''),
                'cosine_distance':f'{float(d):.6f}',
                'preview_180chars':' '.join((doc or '').split())[:180],
                'relevance_grade':'','citation_ok':'','rights_evidence_ok':'','review_notes':''})
print('Đã tạo bảng chấm:', REVIEW_CSV)
print('Số dòng cần chấm:',len(query_rows)*TOP_K)
print('MỞ CSV VÀ CHẤM XONG RỒI MỚI CHẠY PHẦN 5 — KHÔNG CHẠY LẠI Ô NÀY.')

AssertionError: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\data\outputs\retrieval_eval_step4\step4_retrieval_review.csv đã tồn tại. Ô xuất bị khóa để tránh ghi đè nhãn bạn chấm. Chỉ chạy lại ô đánh giá cuối (phần 5).

## 5. Tổng hợp điểm sau khi bạn đã chấm xong CSV

Có thể chạy ngay lần đầu: trả `awaiting_human_labels`, **không tạo điểm đánh giá giả**. Sau khi chấm đủ mọi dòng, chạy **lại riêng ô này**.

**Các metric** (chỉ tính trên 12 câu `in_scope`):
- `Precision@3` = tỷ lệ hit có grade 1 hoặc 2.
- `Strict Hit@3` = tỷ lệ câu có ít nhất một hit grade 2.
- `MRR@3` = trung bình nghịch đảo thứ hạng của hit grade 2 đầu tiên.
- `Mean Grade@3` = điểm trung bình từ 0 đến 2.
- `Citation OK rate` và `Rights evidence status` được báo cáo riêng.

**Không tính Recall@3** vì chưa gán nhãn toàn bộ chunks liên quan trong 66 chunks. Tình huống OOD và an toàn được thống kê riêng, chưa dùng để chốt abstention threshold.

In [7]:
assert REVIEW_CSV.is_file(), 'Chưa có CSV nhãn, hãy chạy ô 4 ở trên'
with REVIEW_CSV.open(encoding='utf-8-sig',newline='') as f:
    labeled=list(csv.DictReader(f))
assert len(labeled)==len(query_rows)*TOP_K, 'Số dòng bảng chấm thay đổi — dừng để kiểm tra'
key_pairs=[(r['query_id'],r['rank']) for r in labeled]
assert len(set(key_pairs))==len(key_pairs), 'Trùng query_id + rank'
all_qids={q['query_id'] for q in query_rows}
assert {r['query_id'] for r in labeled}==all_qids
required_fields=['relevance_grade','citation_ok','rights_evidence_ok']
missing=[(r['query_id'],r['rank'],field) for r in labeled for field in required_fields if not r.get(field,'').strip()]
if missing:
    status='awaiting_human_labels'
    print('BƯỚC 4: Chưa đủ nhãn đánh giá. Còn thiếu',len(missing),'ô trên',len(labeled),'hit.')
    print('Ví dụ cần điền:',missing[:8])
    print('Vui lòng mở:', REVIEW_CSV)
else:
    assert all(r['relevance_grade'].strip() in {'0','1','2'} for r in labeled), 'Grade phải 0/1/2'
    assert all(r['citation_ok'].strip().lower() in {'yes','no'} for r in labeled), 'Citation phải yes/no'
    assert all(r['rights_evidence_ok'].strip().lower() in {'yes','no','unknown'} for r in labeled), 'Rights phải yes/no/unknown'
    by_query=defaultdict(list)
    for r in labeled:
        by_query[r['query_id']].append(r)
    for qid in by_query:
        by_query[qid]=sorted(by_query[qid],key=lambda z:int(z['rank']))
        assert [int(r['rank']) for r in by_query[qid]]==[1,2,3]
    per_query=[]
    for q in query_rows:
        rr=by_query[q['query_id']]
        grades=[int(r['relevance_grade']) for r in rr]
        first_strict=next((i+1 for i,g in enumerate(grades) if g==2),None)
        per_query.append({
            'query_id':q['query_id'],'type':q['query_type'],'language':q['language'],
            'precision_at_3':sum(g>=1 for g in grades)/TOP_K,
            'strict_hit_at_3':int(first_strict is not None),
            'mrr_at_3':1/first_strict if first_strict else 0.,
            'mean_grade_at_3':sum(grades)/TOP_K,
            'graded_hits':grades,
            'top1_source':rr[0]['source_id'],
        })
    core=[a for a in per_query if a['type']=='in_scope']
    assert len(core)==12
    agg=lambda k:round(statistics.mean(r[k] for r in core),4)
    citation_ok=sum(r['citation_ok'].strip().lower()=='yes' for r in labeled)
    rights_counts=dict(Counter(r['rights_evidence_ok'].strip().lower() for r in labeled))
    report={
        'experiment':'EXP-RAG-RETRIEVAL-STEP4-001','stage':'phase4_step4_retrieval_evaluation',
        'created_at_utc':datetime.now(timezone.utc).isoformat(),
        'status':'human_labels_complete_research_only',
        'step3_source_snapshot_sha256':r3['source_snapshot_sha256'],
        'collection':collection.name,'embedding_model':MODEL_ID,'indexed_records_unchanged':collection.count(),
        'queries_total':len(query_rows),'queries_in_scope':len(core),'top_k':TOP_K,
        'metrics_in_scope':{'precision_at_3':agg('precision_at_3'),
            'strict_hit_at_3':agg('strict_hit_at_3'),'mrr_at_3':agg('mrr_at_3'),
            'mean_grade_at_3':agg('mean_grade_at_3')},
        'citation_ok_hits':citation_ok,'citation_checked_hits':len(labeled),
        'rights_evidence_counts':rights_counts,
        'source_reviews_independent':False,
        'per_query':per_query,
        'limitations':['Human labels are local reviewer judgments, not clinical validation',
            'Recall@3 not computed: no exhaustive relevance labels for 66 chunks',
            'Source permissions require independent verification before production',
            'Out-of-scope/safety queries do not establish a calibrated abstention rule'],
        'llm_called':False,'embedding_index_modified':False,'model_retrained':False
    }
    RESULTS_JSON.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    print('===== GIAI ĐOẠN 4 — BƯỚC 4: RETRIEVAL EVALUATION =====')
    print('Collection:',collection.name,'| Chunks:',collection.count())
    print('Questions:',len(query_rows),'| In-scope:',len(core),'| Top-k:',TOP_K)
    for k,v in report['metrics_in_scope'].items():print(k+':',v)
    print('Citation OK:',citation_ok,'/',len(labeled))
    print('Rights evidence:',rights_counts)
    for a in per_query:print(a['query_id'],a['type'],'P@3=',round(a['precision_at_3'],3),'strict hit=',a['strict_hit_at_3'],'top1=',a['top1_source'])
    print('LLM: NO | INDEX MODIFIED: NO | ML TRAINING: NO')
    print('Saved report:', RESULTS_JSON)
    print('===== KẾT THÚC — GỬI OUTPUT VÀ JSON CHO TÔI =====')

===== GIAI ĐOẠN 4 — BƯỚC 4: RETRIEVAL EVALUATION =====
Collection: diabetes_kb_step3_e5small_v1 | Chunks: 66
Questions: 14 | In-scope: 12 | Top-k: 3
precision_at_3: 0.8611
strict_hit_at_3: 0.8333
mrr_at_3: 0.7778
mean_grade_at_3: 1.3889
Citation OK: 25 / 42
Rights evidence: {'no': 29, 'yes': 13}
Q01 in_scope P@3= 0.667 strict hit= 0 top1= LC-STAGES-007
Q02 in_scope P@3= 1.0 strict hit= 1 top1= NIDDK-DIAG-003
Q03 in_scope P@3= 0.667 strict hit= 1 top1= LC-HBA1C-006
Q04 in_scope P@3= 1.0 strict hit= 1 top1= LC-HBA1C-006
Q05 in_scope P@3= 1.0 strict hit= 0 top1= NIDDK-DIAG-003
Q06 in_scope P@3= 0.667 strict hit= 1 top1= LC-STAGES-007
Q07 in_scope P@3= 1.0 strict hit= 1 top1= LC-STAGES-007
Q08 in_scope P@3= 0.667 strict hit= 1 top1= NIDDK-T2D-004
Q09 in_scope P@3= 1.0 strict hit= 1 top1= LC-STAGES-007
Q10 in_scope P@3= 0.667 strict hit= 1 top1= LC-STAGES-007
Q11 in_scope P@3= 1.0 strict hit= 1 top1= NIDDK-T2D-004
Q12 in_scope P@3= 1.0 strict hit= 1 top1= NIDDK-T2D-004
Q13 out_of_scope P@3=